In [ ]:
%cd ..

In [1]:
from dotenv import load_dotenv

load_dotenv()


True

In [ ]:
import sys
import os

sys.path.insert(0, r"C:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions")

In [2]:
import os
import sys
import json
import time
import requests
import pandas as pd
import pyarrow as pa
from dateutil import parser
import pyarrow.parquet as pq
from datetime import datetime


In [3]:

if not os.path.exists("./tmp/data"):
    os.makedirs("./tmp/data")


In [ ]:
from common.config import *
from common.http_util import *
from common.crawler_util import *
from common.ambari_util import *


def fetch_resource_name_freshwork(resource_name, records, **kwargs):
    print("Start crawl : ", resource_name)
    start_time = time.time()

    HDFS_BASE = "s3a://vcs-raw/cx-cso-raw"
    # STATE_PATH = rc["state_path"]
    # BASE_URL = None
    # RESOURCE_URL = None
    # API_KEY_PATH = rc["api_key_path"]
    # API_COOKIE_PATH = rc["api_cookie_path"]
    # QUERY_PARAMS = None
    ENABLE_STATE =False
    HIVE_DB = "cx_cso_raw"
    # crawl_mode = "modified_and_new"
    crawl_mode = kwargs.get("crawl_mode", "static")
    schema_local_path = None
    # result_json_key = "deleted_deals"

    print(resource_name)
    start_time = time.time()
    # =========================
    # MAIN
    # =========================
    if ENABLE_STATE:
        last_state = read_last_state(resource_name)
        print("Last state =", last_state)
    else:
        last_state = None

    if not records:
        print("No new data")
        out_of_data = True
        return True

    # =========================
    # Pandas → Parquet
    # =========================

    now = datetime.now()
    partition_path = "{}/{}".format(HDFS_BASE, resource_name)

    filename = "data_{}_{}{:02d}{:02d}_{}{:02d}{:02d}.parquet".format(
        resource_name, now.year, now.month, now.day, now.hour, now.minute, now.second
    )
    local_parquet = "./tmp/data/cx_cso_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_parquet), exist_ok=True)

    # df.to_parquet(local_parquet,engine="pyarrow", compression="snappy", index=False)
    records = convert_json_add_ts_columns(records)

    schema_tm_path = "./resources/parquet_schema/cx_cso_raw/{}.json".format(resource_name)
    schema = None
    if schema_local_path:
        schema = load_pyarrow_schema_from_json(schema_local_path)

    if os.path.exists(schema_tm_path):
        schema = load_pyarrow_schema_from_json(schema_tm_path)

    if not schema:
        schema = infer_schema_from_json(records)
        schema_json = save_pyarrow_type_to_json(schema)
        write_file_json(schema_tm_path, schema_json)

    records = convert_json_list_by_arrow_schema(records, schema)

    df = pd.DataFrame(records)
    data_table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)

    pq.write_table(data_table, local_parquet, compression="snappy")

    if crawl_mode == "static":
        replace_hdfs_https("{}".format(partition_path), local_parquet)
    else:
        upload_hdfs_https("{}".format(partition_path), local_parquet)

    print("Uploaded parquet to", partition_path)

    # =========================
    # Generate SQL (TEXT ONLY)
    # =========================
    sql = gen_spark_create_table(
        schema=schema,
        db=HIVE_DB,
        table=resource_name,
        location="{}/{}".format(HDFS_BASE, resource_name),
    )

    # filename = "create_table_{}_{}{:02d}{:02d}.sql".format(
    #     resource_name,
    #     now.hour,
    #     now.minute,
    #     now.second
    # )

    filename = "create_table_{}.sql".format(resource_name)

    local_sql = "./tmp/data/cx_cso_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_sql), exist_ok=True)

    with open(local_sql, "w") as f:
        f.write(sql)

    # upload_hdfs_https(
    #     "{}/{}".format(HDFS_BASE, resource_name),
    #     local_sql
    # )

    print("Uploaded SQL definition")

    # =========================
    # Save new state
    # =========================
    if ENABLE_STATE and "updated_at" in df.columns:
        max_ts = get_max_updated_at_str(df)
        print("last state ", resource_name, "max_ts=", max_ts)
        max_ts = subtract_minutes(max_ts, 30)
        write_last_state(max_ts, resource_name)
        print("last state ", resource_name, "max_ts=", max_ts)

    elapsed = time.time() - start_time
    print("Loop {} took {:.3f}s".format(resource_name, elapsed))
    if len(records) < 100:
        print("No new data")
        out_of_data = True
        return True
    return False


In [5]:
import pandas as pd
import re
from collections import defaultdict
def excel_col_name(idx: int) -> str:
    """Zero-based index → Excel column (A, B, ..., AA)"""
    name = ""
    while idx >= 0:
        idx, rem = divmod(idx, 26)
        name = chr(rem + ord("A")) + name
        idx -= 1
    return name


def snake_case(text: str) -> str:
    text = (
        str(text)
        .strip()
        .lower()
        .replace("\n", " ")
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text

def read_excel_and_normalize_columns(
    file_path: str,
    mapping: dict,
    sheet_name=0,
    header_row=0,
    drop_rows=2
) -> pd.DataFrame:
    # đọc raw, chưa set header
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=None, dtype=str)

    raw_headers = df.iloc[header_row]
    seen = defaultdict(int)
    new_columns = []

    for idx, col in enumerate(raw_headers):
        if pd.isna(col) or col == "":
            base = "nan"
        else:
            # 1️⃣ ưu tiên dictionary
            base = mapping.get(col)

            # 2️⃣ fallback snake_case
            if base is None:
                print(f"Not found for col = '{col}'")
                base = snake_case(col)

        seen[base] += 1

        if seen[base] > 1:
            excel_col = excel_col_name(idx).lower()
            base = f"{base}__{excel_col}"

        new_columns.append(base)

    # gán columns sạch
    df.columns = new_columns

    # drop header rows
    df = df.iloc[drop_rows:].reset_index(drop=True)

    return df



In [15]:

COLUMN_DICT  = {
  'Index': 'id',
  'Journey': 'journey_name'
}
resource_name = "cx_dim_journey"

df = read_excel_and_normalize_columns(
    file_path=r"C:\Users\namtv40\Documents\AI Chatbot\CX_def_kpi\CX_def_kpi\cx_dim_journey.xlsx",
    sheet_name="cx_dim_journey",
    mapping=COLUMN_DICT,
    header_row=0,
    drop_rows=1,
)

import json
df = df.fillna("")

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")

with open(f"{resource_name}.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")


✅ Done: cx_dim_journey.json created


In [16]:
fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

Start crawl :  cx_dim_journey
cx_dim_journey
Replace Upload  /opt/datasets/crawlers/vcs/cx-cso-manual/data/cx_dim_journey ./tmp/data/cx_dim_journey/data_cx_dim_journey_20260206_155028.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to /opt/datasets/crawlers/vcs/cx-cso-manual/data/cx_dim_journey
Uploaded SQL definition
Loop cx_dim_journey took 5.351s
No new data


True

In [17]:

COLUMN_DICT  = {
  
}
resource_name = "dim_question_customer_journey"

df = read_excel_and_normalize_columns(
     file_path= r"C:\Users\namtv40\Documents\AI Chatbot\CX_def_kpi\CX_def_kpi\dim_question_customer_journey.xlsx",
    sheet_name="dim_question_customer_journey",
    mapping=COLUMN_DICT,
    header_row=0,
    drop_rows=1,
)

import json
df = df.fillna("")

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")

with open(f"{resource_name}.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")



Not found for col = 'survey_id'
Not found for col = 'survey_name'
Not found for col = 'question_id'
Not found for col = 'question_type'
Not found for col = 'question_text'
Not found for col = 'answer_choice_id'
Not found for col = 'answer_choice_content'
Not found for col = 'question_type'
Not found for col = 'active'
Not found for col = 'kpi'
Not found for col = 'product_category'
✅ Done: dim_question_customer_journey.json created


In [18]:
fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

Start crawl :  dim_question_customer_journey
dim_question_customer_journey
Replace Upload  /opt/datasets/crawlers/vcs/cx-cso-manual/data/dim_question_customer_journey ./tmp/data/dim_question_customer_journey/data_dim_question_customer_journey_20260206_155042.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to /opt/datasets/crawlers/vcs/cx-cso-manual/data/dim_question_customer_journey
Uploaded SQL definition
Loop dim_question_customer_journey took 7.438s


False

In [19]:

COLUMN_DICT  = {
  'Note': 'note'
}
resource_name = "dim_surveys_customer_journey"

df = read_excel_and_normalize_columns(
    file_path=  r"C:\Users\namtv40\Documents\AI Chatbot\CX_def_kpi\CX_def_kpi\dim_surveys_customer_journey.xlsx",
    sheet_name="dim_surveys_customer_journey",
    mapping=COLUMN_DICT,
    header_row=0,
    drop_rows=1,
)

import json
df = df.fillna("")

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")

with open(f"{resource_name}.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")



Not found for col = 'survey_id'
Not found for col = 'survey_name'
Not found for col = 'journey'
Not found for col = 'customer touchpoint'
Not found for col = 'product_category'
Not found for col = 'created_ticket_vs_negative'
✅ Done: dim_surveys_customer_journey.json created


In [20]:
fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

Start crawl :  dim_surveys_customer_journey
dim_surveys_customer_journey
Replace Upload  /opt/datasets/crawlers/vcs/cx-cso-manual/data/dim_surveys_customer_journey ./tmp/data/dim_surveys_customer_journey/data_dim_surveys_customer_journey_20260206_155052.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to /opt/datasets/crawlers/vcs/cx-cso-manual/data/dim_surveys_customer_journey
Uploaded SQL definition
Loop dim_surveys_customer_journey took 9.764s
No new data


True

In [6]:

COLUMN_DICT  = {
  'Note': 'note'
}
resource_name = "cx_product_category"

df = read_excel_and_normalize_columns(
    file_path=  r"C:\Users\namtv40\Projects\Pythons\crawlers\resources\CSO_keymap.xlsx",
    sheet_name="CX_Product_Category",
    mapping=COLUMN_DICT,
    header_row=0,
    drop_rows=1,
)

import json
df = df.fillna("")

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")

with open(f"{resource_name}.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")



Not found for col = 'issue_category'
Not found for col = 'product_category_code'
Not found for col = 'product_category'
✅ Done: cx_product_category.json created


In [7]:
fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

Start crawl :  cx_product_category
cx_product_category
Replace Upload  /opt/datasets/crawlers/vcs/cx-cso-manual/data/cx_product_category ./tmp/data/cx_product_category/data_cx_product_category_20260325_134445.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to /opt/datasets/crawlers/vcs/cx-cso-manual/data/cx_product_category
Uploaded SQL definition
Loop cx_product_category took 1.557s
No new data


True

In [8]:

COLUMN_DICT  = {
  'Note': 'note'
}
resource_name = "cx_company"

df = read_excel_and_normalize_columns(
    file_path=  r"C:\Users\namtv40\Projects\Pythons\crawlers\resources\CSO_keymap.xlsx",
    sheet_name="CX_Company",
    mapping=COLUMN_DICT,
    header_row=0,
    drop_rows=1,
)

import json
df = df.fillna("")

# 6. Chuyển sang JSON
records = df.to_dict(orient="records")

with open(f"{resource_name}.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)

print(f"✅ Done: {resource_name}.json created")



Not found for col = 'id'
Not found for col = 'name'
Not found for col = 'company_alias'
Not found for col = 'tax_code'
Not found for col = 'customer_segment_l1'
Not found for col = 'customer_segment_l2'
Not found for col = 'customer_segment_l3'
✅ Done: cx_company.json created


In [9]:
fetch_resource_name_freshwork(resource_name, records, crawl_mode="static")

Start crawl :  cx_company
cx_company
Replace Upload  /opt/datasets/crawlers/vcs/cx-cso-manual/data/cx_company ./tmp/data/cx_company/data_cx_company_20260325_134451.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/remove None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/fileops/mkdir None
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to /opt/datasets/crawlers/vcs/cx-cso-manual/data/cx_company
Uploaded SQL definition
Loop cx_company took 1.329s


False